# Residual U-Net 2.5D — Direct Segmentation dari µCT Large-Rot-Step

```
RESIDUAL U-NET 2.5D — Direct Segmentation from Noisy Large-Rot-Step µCT
=======================================================================

Arsitektur DISAMAKAN dengan Blok B yang sudah dilatih (apple-to-apple):
  - 2.5D : input 5-channel (slice tengah + 2 atas + 2 bawah), offset -2..+2
  - Residual U-Net FILTERS=[32,64,128,256,512], decoder bilinear Upsample
  - Output 2 kelas (softmax). KONVENSI: class index 0 = PORI.
  - Inferensi pori = softmax(logits)[:, 0]  (identik infer_local_v3.py / notebook
    Segmentasi_Petrofisika_Compare)

Referensi metode: Wang et al. 2024 (SPE J., 2.5D U-Net & variannya untuk
segmentasi pori/DFN pada digital rock). Model menggunakan 5 citra 2D bertumpuk
(central + 2 atas + 2 bawah) sebagai konteks pseudo-3D, output peta segmentasi
diproses softmax per-pixel lalu argmax/threshold.

Strategi I/O:
  INPUT  : large rot.step (0.8° dan/atau 0.4°) — noisy, mengandung ring artifacts,
           BELUM tersegmentasi (grayscale), dinormalisasi [0,1].
  TARGET : small rot.step (0.2°) — minim noise, tanpa ring, SUDAH BINARY (pori=1).
  OUTPUT : peta probabilitas pori [0,1] = softmax(...)[class 0].

Yang DISAMAKAN dengan pipeline RED-CNN agar perbandingan adil:
  - Split per-slice 80/10/10 (split_slices identik)
  - Patch 256x256, 20 patch/slice acak, augmentasi rot90+flip identik
  - Normalisasi INPUT pakai percentile 1-99 identik (target tetap biner)
  - Inference patch-based + Hanning blending identik
  - ResourceTracker, AMP, DataLoader workers identik
  - Petrophysics paralel (porositas, SSA, Kozeny-Carman) identik

DUA jalur evaluasi dari model yang SAMA:
  (1) SEGMENTASI LANGSUNG : prob_pori >= 0.5  -> biner   (jalur native U-Net)
  (2) REGRESI->OTSU        : prob_pori di-Otsu -> biner   (apple-to-apple persis
                            dengan cara RED-CNN menghasilkan biner)

Dependensi:
  pip install torch torchvision tifffile scikit-image scipy \
              matplotlib pandas tqdm psutil pynvml
```

## 0. Setup Colab — Mount Drive & Install Dependencies
Jalankan sekali di awal sesi. Pilih **Runtime → Change runtime type → GPU (L4 disarankan)** terlebih dahulu.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies (torch sudah tersedia di Colab)
!pip install -q tifffile scikit-image scipy matplotlib pandas tqdm psutil pynvml

# Cek GPU yang aktif
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Import & Device Setup

In [ ]:
# ============================================================================
# CELL 1: IMPORT
# ============================================================================
import os, gc, time, json, warnings, threading, multiprocessing
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from datetime import datetime
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast

from tifffile import imread, imwrite
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_erosion
import psutil

warnings.filterwarnings('ignore')

CPU_COUNT = multiprocessing.cpu_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

## 2. ResourceTracker (monitor CPU/GPU/RAM/VRAM)

In [ ]:
# ============================================================================
# RESOURCE TRACKER  (identik dengan pipeline RED-CNN)
# ============================================================================
class ResourceTracker:
    def __init__(self, interval: float = 2.0, log_path: str = "resource_log.csv"):
        self.interval = interval
        self.log_path = log_path
        self._stop_evt = threading.Event()
        self._lock = threading.Lock()
        self._phase = "idle"
        self._records = []
        self._thread = threading.Thread(target=self._run, daemon=True, name="ResourceTracker")
        self._nvml_ok = False
        try:
            import pynvml
            pynvml.nvmlInit()
            self._nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            self._pynvml = pynvml
            self._nvml_ok = True
            print("ResourceTracker: GPU monitoring via pynvml ✓")
        except Exception:
            try:
                import GPUtil
                self._gputil = GPUtil
                print("ResourceTracker: GPU monitoring via GPUtil ✓")
            except Exception:
                print("ResourceTracker: GPU monitoring tidak tersedia")

    def start(self):
        self._thread.start()
        print(f"ResourceTracker: started (interval={self.interval}s, log={self.log_path})")

    def stop(self):
        self._stop_evt.set()
        self._thread.join(timeout=5)
        self._flush_csv()
        print(f"ResourceTracker: stopped — {len(self._records)} records → {self.log_path}")

    def set_phase(self, phase):
        with self._lock: self._phase = phase

    @contextmanager
    def phase(self, name):
        self.set_phase(name)
        print(f"\n[ResourceTracker] ── Phase: {name} ──")
        try: yield
        finally: self.set_phase("idle")

    def _gpu_stats(self):
        gpu_util = vram_used_mb = vram_total_mb = 0.0
        if self._nvml_ok:
            try:
                mem = self._pynvml.nvmlDeviceGetMemoryInfo(self._nvml_handle)
                util = self._pynvml.nvmlDeviceGetUtilizationRates(self._nvml_handle)
                gpu_util = float(util.gpu); vram_used_mb = mem.used/1e6; vram_total_mb = mem.total/1e6
            except Exception: pass
        elif hasattr(self, '_gputil'):
            try:
                gpus = self._gputil.getGPUs()
                if gpus:
                    gpu_util = gpus[0].load*100; vram_used_mb = gpus[0].memoryUsed; vram_total_mb = gpus[0].memoryTotal
            except Exception: pass
        elif device.type == 'cuda':
            vram_used_mb = torch.cuda.memory_allocated()/1e6
            vram_total_mb = torch.cuda.get_device_properties(0).total_memory/1e6
        return gpu_util, vram_used_mb, vram_total_mb

    def _run(self):
        while not self._stop_evt.is_set():
            ts = datetime.now().isoformat(timespec='seconds')
            cpu_total = psutil.cpu_percent(interval=None)
            ram = psutil.virtual_memory()
            n_threads = threading.active_count()
            gpu_util, vram_used, vram_total = self._gpu_stats()
            with self._lock:
                self._records.append({
                    'timestamp': ts, 'phase': self._phase, 'cpu_total_%': cpu_total,
                    'ram_used_mb': ram.used/1e6, 'ram_total_mb': ram.total/1e6, 'ram_%': ram.percent,
                    'gpu_util_%': gpu_util, 'vram_used_mb': vram_used, 'vram_total_mb': vram_total,
                    'active_threads': n_threads,
                })
            self._stop_evt.wait(self.interval)

    def _flush_csv(self):
        if self._records: pd.DataFrame(self._records).to_csv(self.log_path, index=False)

    def summary(self):
        if not self._records: return
        df = pd.DataFrame(self._records)
        print("\n" + "="*65); print("  RESOURCE SUMMARY PER PHASE"); print("="*65)
        for phase, grp in df.groupby('phase'):
            print(f"\n  [{phase.upper()}]")
            print(f"    Duration    : {len(grp)*self.interval:.0f}s  ({len(grp)} samples)")
            print(f"    CPU avg/max : {grp['cpu_total_%'].mean():.1f}% / {grp['cpu_total_%'].max():.1f}%")
            print(f"    RAM avg/max : {grp['ram_used_mb'].mean()/1024:.2f} / {grp['ram_used_mb'].max()/1024:.2f} GB")
            print(f"    GPU avg/max : {grp['gpu_util_%'].mean():.1f}% / {grp['gpu_util_%'].max():.1f}%")
            print(f"    VRAM avg/max: {grp['vram_used_mb'].mean()/1024:.2f} / {grp['vram_used_mb'].max()/1024:.2f} GB")

    def plot(self, save_path="resource_usage.png"):
        if not self._records: return
        df = pd.DataFrame(self._records); df['t'] = range(len(df))
        fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
        fig.suptitle('Resource Usage — Residual U-Net Pipeline', fontsize=13, fontweight='bold')
        axes[0].plot(df['t'], df['cpu_total_%'], color='steelblue', lw=1.5); axes[0].set_ylabel('CPU %'); axes[0].set_ylim(0,105); axes[0].grid(alpha=0.3)
        axes[1].plot(df['t'], df['ram_used_mb']/1024, color='purple', lw=1.5); axes[1].set_ylabel('RAM GB'); axes[1].grid(alpha=0.3)
        axes[2].plot(df['t'], df['gpu_util_%'], color='tomato', lw=1.5); axes[2].set_ylabel('GPU %'); axes[2].set_ylim(0,105); axes[2].grid(alpha=0.3)
        axes[3].plot(df['t'], df['vram_used_mb']/1024, color='darkorange', lw=1.5); axes[3].set_ylabel('VRAM GB'); axes[3].set_xlabel(f'Sample (interval={self.interval}s)'); axes[3].grid(alpha=0.3)
        plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close()
        print(f"Resource plot saved: {save_path}")

## 3. Konfigurasi

**Sesuaikan path & parameter di sini sebelum run.** Perhatikan `PATH_SEG`, `PORE_IS_LOW`, dan `TRAINING_PAIR`.

In [ ]:
# ============================================================================
# CELL 2: KONFIGURASI
# ============================================================================
class Config:
    # ── PATH (Google Drive). Sesuaikan dengan struktur Anda ──────────────────
    PATH_02 = '/content/drive/MyDrive/Dataset TA/Libo/Training di lab/Training Data/Libo 512/ground_truth_crop_z200-712_y200-712_x200-712.tif'  # 0.2° clean
    PATH_04 = '/content/drive/MyDrive/Dataset TA/Libo/Training di lab/Training Data/Libo 512/poor_04_crop_z200-712_y200-712_x200-712.tif'         # 0.4° noisy
    PATH_08 = '/content/drive/MyDrive/Dataset TA/Libo/Training di lab/Training Data/Libo 512/poor_08_crop_z200-712_y200-712_x200-712.tif'         # 0.8° noisy

    # Target SEGMENTED. Jika Anda PUNYA file biner small-rot-step terpisah, isi path-nya:
    #   PATH_SEG = '/content/drive/MyDrive/TA/Libo512/seg_02_binary.tif'   (pori=1)
    # Jika None → target biner dibuat otomatis dari PATH_02 via Otsu (pori = nilai rendah).
    PATH_SEG = '/content/drive/MyDrive/Dataset TA/Libo/Training di lab/Training Data/Libo 512/ground_truth_seg_crop_z200-712_y200-712_x200-712.tif'

    # 'both' = input 0.8° dan 0.4° digabung sebagai dua sumber noisy (target sama).
    # '0.8'  = hanya 0.8°.   '0.4' = hanya 0.4°.
    TRAINING_PAIR = 'both'

    PORE_IS_LOW = True       # True: pori = nilai grayscale RENDAH (gelap) → Otsu pakai (vol < thresh)

    CROP_SIZE = None
    CROP_ORIGIN = None

    TRAIN_FRAC = 0.80
    VAL_FRAC   = 0.10

    PATCH_SIZE        = 256
    PATCHES_PER_SLICE = 20
    STRIDE_TEST       = 96

    # ── Best params Res-UNet (Colab Pro) ─────────────────────────────────────
    # Catatan: arsitektur Res-UNet 2.5D memakai FILTERS tetap [32,64,128,256,512]
    # (identik Blok B). BASE_FILTERS/DEPTH di bawah TIDAK lagi mengubah model,
    # hanya dipertahankan utk kompatibilitas log lama.
    BASE_FILTERS = 32          # (informasi: filter stage-1 = 32)
    DEPTH        = 4           # (informasi: 4 stage enc/dec)
    BATCH_SIZE   = 16          # 256² patch + U-Net depth-4 @64f → aman utk 16 GB; naikkan ke 24-32 di A100
    EPOCHS       = 120
    LR           = 2e-4        # U-Net + CE/Dice; sesuaikan bila val loss tak turun
    LR_MIN       = 1e-6
    PATIENCE_ES  = 20
    PATIENCE_LR  = 8
    LR_FACTOR    = 0.5
    WEIGHT_DECAY = 1e-5

    # ── Arsitektur 2.5D (DISAMAKAN dengan Blok B) ────────────────────────────
    UNET_IN_CH  = 5            # 2.5D: central + 2 atas + 2 bawah
    N_CLASSES   = 2            # softmax 2 kelas; class index 0 = PORI

    # Loss segmentasi: Dice + CrossEntropy (multikelas)
    CE_WEIGHT   = 0.5          # total = CE_WEIGHT*CE + (1-CE_WEIGHT)*Dice
    AUGMENT     = True

    # ── Petrophysics ─────────────────────────────────────────────────────────
    VOXEL_SIZE_UM   = 5.714
    KC_TORTUOSITY   = 2.5
    KC_SHAPE_FACTOR = 2.0

    SAVE_DIR   = '/content/drive/MyDrive/Dataset TA/Res-Net_Seg_2'
    MODEL_NAME = 'resunet_seg_best.pth'

    USE_AMP     = True         # Colab GPU → AMP ON (RED-CNN Anda OFF di lokal; di Colab nyalakan)
    USE_COMPILE = False        # torch.compile bisa rewel di Colab; biarkan OFF dulu
    NUM_WORKERS = 2            # Colab umumnya 2 vCPU
    TRACKER_INTERVAL = 2.0

cfg = Config()

## 4. Arsitektur Residual U-Net 2.5D (identik Blok B)

Arsitektur **persis** dengan model segmentasi yang sudah dilatih
(`Segmentasi_Petrofisika_Compare` / `infer_local_v3.py`):
`in_ch=5` (2.5D), `FILTERS=[32,64,128,256,512]`, decoder bilinear `Upsample`,
shortcut residual `Conv1x1+BN`, output **2 kelas** (softmax, class 0 = pori).

In [ ]:
# ============================================================================
# CELL 3: ARSITEKTUR RESIDUAL U-NET 2.5D
# ---------------------------------------------------------------------------
# DISALIN PERSIS dari notebook Blok B (Segmentasi_Petrofisika_Compare) agar
# bobot/arsitektur konsisten dan perbandingan K1/K2/K3 apple-to-apple.
#   in_ch = 5 (2.5D)        FILTERS = [32,64,128,256,512]
#   decoder: bilinear Upsample + ResidualBlock(concat)
#   output : 2 kelas (softmax). class index 0 = PORI.
# ============================================================================
import torch.nn.functional as F


class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.shortcut = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                                       nn.BatchNorm2d(out_ch))
                         if in_ch != out_ch else nn.Identity())

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = ResidualBlock(in_ch, out_ch)
        self.pool  = nn.MaxPool2d(2, 2)

    def forward(self, x):
        skip = self.block(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.block = ResidualBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        return self.block(torch.cat([x, skip], dim=1))


class ResidualUNet(nn.Module):
    """Residual U-Net 2.5D. in_ch=5, output 2 kelas (logits). class 0 = pori."""
    FILTERS = [32, 64, 128, 256, 512]

    def __init__(self, in_ch=5, n_classes=2):
        super().__init__()
        f = self.FILTERS
        self.enc1 = EncoderBlock(in_ch, f[0]); self.enc2 = EncoderBlock(f[0], f[1])
        self.enc3 = EncoderBlock(f[1], f[2]);  self.enc4 = EncoderBlock(f[2], f[3])
        self.bottleneck = ResidualBlock(f[3], f[4])
        self.dec4 = DecoderBlock(f[4], f[3], f[3]); self.dec3 = DecoderBlock(f[3], f[2], f[2])
        self.dec2 = DecoderBlock(f[2], f[1], f[1]); self.dec1 = DecoderBlock(f[1], f[0], f[0])
        self.out  = nn.Conv2d(f[0], n_classes, kernel_size=1)

    def forward(self, x):
        x, s1 = self.enc1(x); x, s2 = self.enc2(x)
        x, s3 = self.enc3(x); x, s4 = self.enc4(x)
        x = self.bottleneck(x)
        x = self.dec4(x, s4); x = self.dec3(x, s3)
        x = self.dec2(x, s2); x = self.dec1(x, s1)
        return self.out(x)            # logits (B, 2, H, W)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 5. Data Loading & Patch Dataset

In [ ]:
# ============================================================================
# CELL 4: DATA LOADING (kerangka RED-CNN; input 2.5D 5-ch, target biner)
# ============================================================================
def load_and_crop_volume(tif_path, crop_size, crop_origin):
    print(f"  Loading: {os.path.basename(tif_path)} ...", end='', flush=True)
    vol = imread(tif_path).astype(np.float32)
    print(f" shape={vol.shape}")
    if crop_size is not None:
        D, H, W = vol.shape; cs = crop_size
        if crop_origin is not None: x0, y0, z0 = crop_origin
        else: x0 = max(0,(W-cs)//2); y0 = max(0,(H-cs)//2); z0 = max(0,(D-cs)//2)
        x0 = min(x0,W-cs); y0 = min(y0,H-cs); z0 = min(z0,D-cs)
        vol = vol[z0:z0+cs, y0:y0+cs, x0:x0+cs]
        print(f"  -> Cropped -> {vol.shape}")
    return vol


def normalize_volume(vol, p1=None, p99=None):
    """Normalisasi grayscale 1-99 percentile (identik RED-CNN). Untuk INPUT saja."""
    if p1  is None: p1  = np.percentile(vol, 1)
    if p99 is None: p99 = np.percentile(vol, 99)
    vol_n = np.clip((vol - p1) / (p99 - p1 + 1e-8), 0.0, 1.0)
    return vol_n.astype(np.float32), p1, p99


def binarize_grayscale(vol, pore_is_low=True, method='otsu'):
    """Buat target biner dari grayscale 0.2deg. pori=1."""
    flat = vol.flatten()
    thresh = threshold_otsu(flat) if method == 'otsu' else np.percentile(flat, 30)
    bin_vol = (vol < thresh) if pore_is_low else (vol > thresh)
    return bin_vol.astype(np.float32), float(thresh)


def split_slices(n_slices, train_frac, val_frac):
    n_train = int(n_slices * train_frac)
    n_val   = int(n_slices * val_frac)
    return (list(range(0, n_train)),
            list(range(n_train, n_train + n_val)),
            list(range(n_train + n_val, n_slices)))


def stack_2p5d(vol, z, n_blocks=1, block_len=None):
    """Ambil jendela 5-channel (offset -2..+2) di sekitar slice z.

    Penting saat input gabungan 0.8deg+0.4deg ditumpuk di sumbu Z: jendela tidak
    boleh menembus batas antar-blok. n_blocks & block_len membatasi clamp di
    dalam blok asal slice z.
    """
    if block_len is None:
        lo, hi = 0, vol.shape[0] - 1
    else:
        b = z // block_len
        lo, hi = b * block_len, (b + 1) * block_len - 1
    idxs = [min(max(z + off, lo), hi) for off in (-2, -1, 0, 1, 2)]
    return np.stack([vol[i] for i in idxs], axis=0)   # (5, H, W)


class SegPatchDataset(Dataset):
    """
    noisy_vol : grayscale input ternormalisasi (bisa concat 0.8deg+0.4deg di Z)
    seg_vol    : target biner {0,1} (pori=1), sejajar slice dgn noisy_vol
    Output  X : (5, ps, ps) jendela 2.5D ternormalisasi
            Y : (ps, ps) long {0=pori, 1=solid}  -> cocok CrossEntropy, class0=pori
    block_len : panjang satu volume sumber (utk clamp 2.5D antar-blok). None=full.
    """
    def __init__(self, noisy_vol, seg_vol, slice_indices, patch_size,
                 patches_per_slice=20, augment=False, mode='train', block_len=None):
        self.noisy = noisy_vol
        self.seg   = seg_vol
        self.patch = patch_size
        self.augment = augment
        self.mode  = mode
        self.patches_per_slice = patches_per_slice
        self.block_len = block_len
        if mode == 'train':
            self.slice_indices = slice_indices
            self.length = len(slice_indices) * patches_per_slice
        else:
            stride = max(patch_size // 2, 32)
            self.coords = []
            _, H, W = noisy_vol.shape
            for z in slice_indices:
                for y in range(0, H - patch_size + 1, stride):
                    for x in range(0, W - patch_size + 1, stride):
                        self.coords.append((z, y, x))
            self.length = len(self.coords)

    def __len__(self): return self.length

    def _augment(self, n, c):
        # n: (5,ps,ps)  c: (ps,ps)
        k = np.random.randint(0, 4)
        n = np.rot90(n, k, axes=(1, 2)); c = np.rot90(c, k)
        if np.random.rand() > 0.5: n = n[:, :, ::-1]; c = c[:, ::-1]
        if np.random.rand() > 0.5: n = n[:, ::-1, :]; c = c[::-1, :]
        return np.ascontiguousarray(n), np.ascontiguousarray(c)

    def __getitem__(self, idx):
        ps = self.patch; _, H, W = self.noisy.shape
        if self.mode == 'train':
            slice_idx = self.slice_indices[idx // self.patches_per_slice]
            y = np.random.randint(0, H - ps); x = np.random.randint(0, W - ps)
        else:
            slice_idx, y, x = self.coords[idx]
        win = stack_2p5d(self.noisy, slice_idx, block_len=self.block_len)  # (5,H,W)
        n_p = win[:, y:y+ps, x:x+ps]                                       # (5,ps,ps)
        c_p = self.seg[slice_idx, y:y+ps, x:x+ps]                          # (ps,ps) {0,1} pori=1
        if self.augment and self.mode == 'train':
            n_p, c_p = self._augment(n_p, c_p)
        # target long: class0=pori, class1=solid  ->  label = 1 - pori
        tgt = (1 - c_p).astype(np.int64)
        return (torch.from_numpy(n_p.copy()).float(),
                torch.from_numpy(tgt.copy()).long())

## 6. Loss Function (Dice + CrossEntropy, multikelas)

In [ ]:
# ============================================================================
# CELL 5: LOSS (Dice + CrossEntropy) — segmentasi 2-kelas (softmax)
# Target: long {0=pori, 1=solid}. Dice dihitung pada kelas pori (class 0).
# ============================================================================
class DiceLossMC(nn.Module):
    """Soft-Dice multikelas dari softmax. Rata-rata Dice antar kelas."""
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, target_long):
        n_cls = logits.shape[1]
        prob = torch.softmax(logits, dim=1)                      # (B,C,H,W)
        oh = F.one_hot(target_long, n_cls).permute(0, 3, 1, 2).float()
        p = prob.reshape(prob.shape[0], n_cls, -1)
        t = oh.reshape(oh.shape[0], n_cls, -1)
        inter = (p * t).sum(-1)
        dice = (2*inter + self.smooth) / (p.sum(-1) + t.sum(-1) + self.smooth)
        return 1.0 - dice.mean()


class DiceCELoss(nn.Module):
    def __init__(self, ce_weight=0.5):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss()
        self.dice = DiceLossMC()
        self.w = ce_weight
    def forward(self, logits, target_long):
        return self.w * self.ce(logits, target_long) + (1 - self.w) * self.dice(logits, target_long)

## 7. Metrik Segmentasi

In [ ]:
# ============================================================================
# CELL 6: METRIK SEGMENTASI
# ============================================================================
def seg_metrics_from_prob(prob, target_bin, thr=0.5):
    """Dice, IoU, Accuracy dari peta probabilitas vs target biner."""
    pred = (prob >= thr).astype(bool)
    t = target_bin.astype(bool)
    tp = np.logical_and(pred, t).sum()
    fp = np.logical_and(pred, ~t).sum()
    fn = np.logical_and(~pred, t).sum()
    tn = np.logical_and(~pred, ~t).sum()
    dice = 2*tp / (2*tp + fp + fn + 1e-8)
    iou  = tp / (tp + fp + fn + 1e-8)
    acc  = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    return {'Dice': float(dice), 'IoU': float(iou), 'Acc': float(acc),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn)}

## 8. Training & Plot History

In [ ]:
# ============================================================================
# CELL 7: TRAINING
# ============================================================================
PORE_CLASS_IDX = 0    # konvensi: class 0 = pori (sama dgn Blok B)


def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train(); total = 0.0
    for noisy, tgt in loader:
        noisy = noisy.to(device, non_blocking=True)
        tgt   = tgt.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.USE_AMP):
            loss = criterion(model(noisy), tgt)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval(); total = 0.0; dice_total = 0.0
    for noisy, tgt in loader:
        noisy = noisy.to(device, non_blocking=True)
        tgt   = tgt.to(device, non_blocking=True)
        with autocast(enabled=cfg.USE_AMP):
            logits = model(noisy)
            total += criterion(logits, tgt).item()
        prob_pore = torch.softmax(logits, dim=1)[:, PORE_CLASS_IDX]   # (B,H,W)
        pred = (prob_pore >= 0.5).float()
        gt   = (tgt == PORE_CLASS_IDX).float()
        inter = (pred * gt).sum()
        dice_total += (2*inter / (pred.sum() + gt.sum() + 1e-8)).item()
    return total / len(loader), dice_total / len(loader)


def train_model(cfg, tracker, train_loader, val_loader, norm_stats):
    model = ResidualUNet(in_ch=cfg.UNET_IN_CH, n_classes=cfg.N_CLASSES).to(device)
    print(f"  Model parameters: {count_parameters(model):,}")
    if cfg.USE_COMPILE:
        try: model = torch.compile(model); print("  torch.compile: OK")
        except Exception as e: print(f"  torch.compile gagal ({e})")

    criterion = DiceCELoss(cfg.CE_WEIGHT).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, 'min', cfg.LR_FACTOR, cfg.PATIENCE_LR, min_lr=cfg.LR_MIN)
    scaler = GradScaler(enabled=cfg.USE_AMP)

    best_val = float('inf'); pat_cnt = 0
    history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'lr': []}
    model_path = os.path.join(cfg.SAVE_DIR, cfg.MODEL_NAME)

    print(f"\n  Training started : {datetime.now():%Y-%m-%d %H:%M:%S}")
    print(f"  AMP={cfg.USE_AMP}  Save={model_path}\n")
    t_start = time.time()

    with tracker.phase("training"):
        for epoch in range(1, cfg.EPOCHS + 1):
            t_ep = time.time()
            tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
            vl_loss, vl_dice = validate(model, val_loader, criterion, device)
            scheduler.step(vl_loss)
            lr_now = optimizer.param_groups[0]['lr']
            history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
            history['val_dice'].append(vl_dice); history['lr'].append(lr_now)

            if vl_loss < best_val:
                best_val = vl_loss; pat_cnt = 0
                torch.save({
                    'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'val_loss': best_val,
                    'val_dice': vl_dice,
                    'config': {k: v for k, v in cfg.__dict__.items() if not k.startswith('__')},
                    'norm_stats': norm_stats,
                }, model_path)
                marker = " <- best"
            else:
                pat_cnt += 1; marker = f" (patience {pat_cnt}/{cfg.PATIENCE_ES})"

            if epoch % 5 == 0 or epoch == 1:
                elapsed = (time.time() - t_start)/60; ep_time = time.time()-t_ep
                avg_ep = elapsed/epoch*60; remain = avg_ep*(cfg.EPOCHS-epoch)/60
                print(f"  Ep {epoch:4d}/{cfg.EPOCHS}  tr={tr_loss:.5f}  vl={vl_loss:.5f}  "
                      f"vDice={vl_dice:.4f}  lr={lr_now:.2e}  {ep_time:.1f}s/ep  "
                      f"ETA~{remain:.1f}min{marker}")

            if pat_cnt >= cfg.PATIENCE_ES:
                print(f"\n  Early stopping at epoch {epoch}"); break

    print(f"\n  Training finished : {datetime.now():%Y-%m-%d %H:%M:%S}")
    print(f"  Total time : {(time.time()-t_start)/60:.1f} min  Best val loss : {best_val:.6f}")
    return model, history, model_path


def plot_training_history(history, save_dir):
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    epochs = range(1, len(history['train_loss'])+1)
    axes[0].plot(epochs, history['train_loss'], label='Train', color='royalblue')
    axes[0].plot(epochs, history['val_loss'],   label='Val',   color='tomato')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (Dice+CE)')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs, history['val_dice'], color='forestgreen')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Dice (pori)'); axes[1].set_title('Validation Dice'); axes[1].grid(alpha=0.3)
    axes[2].semilogy(epochs, history['lr'], color='purple')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR'); axes[2].set_title('Learning Rate'); axes[2].grid(alpha=0.3)
    plt.tight_layout(); p = os.path.join(save_dir, 'training_history.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close(); print(f"  Saved: {p}")

## 9. Inference (patch-based + Hanning blending)

In [ ]:
# ============================================================================
# CELL 8: INFERENCE — 2.5D patch-based + Hanning blending (identik RED-CNN)
# Output = peta PROBABILITAS pori [0,1] = softmax(logits)[class 0]
# ============================================================================
@torch.no_grad()
def _inference_single_slice(args):
    model, win5, device, patch_size, stride = args   # win5: (5,H,W) jendela 2.5D
    model.eval()
    _, H, W = win5.shape
    out = np.zeros((H, W), np.float64); wgt = np.zeros((H, W), np.float64)
    g = np.outer(np.hanning(patch_size), np.hanning(patch_size)) + 1e-6
    ys = list(range(0, H - patch_size + 1, stride))
    xs = list(range(0, W - patch_size + 1, stride))
    if not ys or ys[-1] + patch_size < H: ys.append(max(0, H - patch_size))
    if not xs or xs[-1] + patch_size < W: xs.append(max(0, W - patch_size))
    for y in ys:
        for x in xs:
            patch = win5[:, y:y+patch_size, x:x+patch_size]
            t = torch.from_numpy(patch[None]).float().to(device)        # (1,5,ps,ps)
            with autocast(enabled=cfg.USE_AMP):
                logits = model(t)
                p = torch.softmax(logits, dim=1)[0, PORE_CLASS_IDX].cpu().float().numpy()
            out[y:y+patch_size, x:x+patch_size] += p * g
            wgt[y:y+patch_size, x:x+patch_size] += g
    return np.clip(out / wgt, 0, 1).astype(np.float32)


def run_inference_testset(model, vol_noisy, seg_target, test_idx, cfg, device, tracker,
                          block_len=None):
    print(f"\n  Running inference on {len(test_idx)} test slices ...")
    model.eval()
    prob_slices = [None] * len(test_idx)
    n_workers = min(4, CPU_COUNT)
    with tracker.phase("inference"):
        # bangun jendela 2.5D per slice test (clamp dalam blok asal)
        args_list = [(model, stack_2p5d(vol_noisy, z, block_len=block_len),
                      device, cfg.PATCH_SIZE, cfg.STRIDE_TEST) for z in test_idx]
        with ThreadPoolExecutor(max_workers=n_workers) as pool:
            futures = {pool.submit(_inference_single_slice, a): i for i, a in enumerate(args_list)}
            for fut in tqdm(as_completed(futures), total=len(futures), desc='  Test inference'):
                prob_slices[futures[fut]] = fut.result()
    prob_vol = np.stack(prob_slices, axis=0)

    # ── Metrik regresi (prob vs target biner pori) — apple-to-apple field metrics ──
    seg_test = seg_target[test_idx]
    psnr_list = [psnr_fn(seg_test[i], prob_vol[i], data_range=1.0) for i in range(len(test_idx))]
    ssim_list = [ssim_fn(seg_test[i], prob_vol[i], data_range=1.0) for i in range(len(test_idx))]
    rmse_list = [float(np.sqrt(np.mean((seg_test[i]-prob_vol[i])**2))) for i in range(len(test_idx))]
    reg_metrics = {'PSNR_mean': float(np.mean(psnr_list)), 'PSNR_std': float(np.std(psnr_list)),
                   'SSIM_mean': float(np.mean(ssim_list)), 'SSIM_std': float(np.std(ssim_list)),
                   'RMSE_mean': float(np.mean(rmse_list)), 'RMSE_std': float(np.std(rmse_list))}
    print(f"\n  -- Regression-style metrics (prob vs binary GT) --")
    print(f"  PSNR : {reg_metrics['PSNR_mean']:.4f} +/- {reg_metrics['PSNR_std']:.4f} dB")
    print(f"  SSIM : {reg_metrics['SSIM_mean']:.4f} +/- {reg_metrics['SSIM_std']:.4f}")
    print(f"  RMSE : {reg_metrics['RMSE_mean']:.6f} +/- {reg_metrics['RMSE_std']:.6f}")
    return prob_vol, reg_metrics, psnr_list, ssim_list, rmse_list

## 10. Petrophysical Analysis

In [ ]:
# ============================================================================
# CELL 9: PETROPHYSICS (identik RED-CNN)
# ============================================================================
def calculate_porosity(binary_vol):
    per_sl = binary_vol.mean(axis=(1, 2))
    return {'porosity_total': float(binary_vol.mean()),
            'porosity_percent': float(binary_vol.mean()*100),
            'pore_voxels': int(binary_vol.sum()), 'total_voxels': int(binary_vol.size),
            'porosity_per_slice': per_sl}


def calculate_dice_iou(pred_bin, true_bin):
    p = pred_bin.astype(bool); r = true_bin.astype(bool)
    tp = np.logical_and(p,r).sum(); fp = np.logical_and(p,~r).sum(); fn = np.logical_and(~p,r).sum()
    return {'Dice': float(2*tp/(2*tp+fp+fn+1e-8)), 'IoU': float(tp/(tp+fp+fn+1e-8)),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn)}


def _process_single_slice(args):
    z_idx, bin_sl, phi_frac, voxel_um, tortuosity, shape_factor = args
    vox_m = voxel_um*1e-6; H, W = bin_sl.shape
    surf = bin_sl.astype(int) - binary_erosion(bin_sl).astype(int)
    sa_m2 = surf.sum()*vox_m**2; vol_m3 = H*W*vox_m**3
    Sv_m = sa_m2/vol_m3 if vol_m3 > 0 else 0.0
    if 0 < phi_frac < 1 and Sv_m > 0:
        K_m2 = phi_frac**3 / (shape_factor*tortuosity**2*Sv_m**2*(1-phi_frac)**2)
        K_mD = K_m2 / 9.869233e-16
    else: K_mD = float('nan')
    return z_idx, Sv_m, K_mD


def run_petrophysics_parallel(bin_vol, por, voxel_um, tortuosity, shape_factor, label, tracker):
    n = bin_vol.shape[0]; phi_per_slice = por['porosity_per_slice']
    args_list = [(z, bin_vol[z], float(phi_per_slice[z]), voxel_um, tortuosity, shape_factor) for z in range(n)]
    Sv_arr = np.zeros(n); K_mD_arr = np.full(n, float('nan'))
    n_workers = max(1, CPU_COUNT - 1)
    print(f"  [{label}] ProcessPoolExecutor: {n_workers} workers")
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_process_single_slice, a): a[0] for a in args_list}
        for fut in tqdm(as_completed(futures), total=n, desc=f'  Petrophysics [{label}]'):
            z_idx, Sv_m, K_mD = fut.result(); Sv_arr[z_idx] = Sv_m; K_mD_arr[z_idx] = K_mD
    return Sv_arr, K_mD_arr


def calculate_ssa_volume(binary_vol, voxel_um):
    vox_m = voxel_um*1e-6; D, H, W = binary_vol.shape
    vol_m3 = D*H*W*vox_m**3
    surface = binary_vol.astype(int) - binary_erosion(binary_vol).astype(int)
    ssa = surface.sum()*vox_m**2/vol_m3
    return {'surface_voxels': int(surface.sum()), 'SSA_per_um': float(ssa),
            'SSA_m2_per_cm3': float(ssa*1e4)}


def kozeny_carman_volume(phi_frac, Sv_m, tortuosity, shape_factor):
    if phi_frac <= 0 or phi_frac >= 1 or Sv_m <= 0:
        return {'K_m2': float('nan'), 'K_mD': float('nan')}
    K_m2 = phi_frac**3 / (shape_factor*tortuosity**2*Sv_m**2*(1-phi_frac)**2)
    return {'K_m2': float(K_m2), 'K_mD': float(K_m2/9.869233e-16)}

## 11. Visualisasi

In [ ]:
# ============================================================================
# CELL 10: VISUALISASI
# ============================================================================
def visualize_results(test_noisy, prob_pred, seg_gt,
                      bin_seg05, bin_otsu, bin_gt,
                      seg_m_05, seg_m_otsu,
                      por_gt, por_05, por_otsu,
                      save_dir):
    n_test = len(seg_gt); mid = n_test // 2
    fig, axes = plt.subplots(1, 5, figsize=(26, 6))
    fig.suptitle(f'Residual U-Net Segmentation — Slice tengah (z={mid})', fontsize=12, fontweight='bold')
    for ax, img, ttl, cmap in [
        (axes[0], test_noisy[mid], 'Input noisy (0.8°/0.4°)', 'gray'),
        (axes[1], prob_pred[mid],  'Prob pori (U-Net)', 'viridis'),
        (axes[2], bin_seg05[mid].astype(float), 'Biner: thr=0.5', 'binary_r'),
        (axes[3], bin_otsu[mid].astype(float),  'Biner: Otsu(prob)', 'binary_r'),
        (axes[4], bin_gt[mid].astype(float),    'GT biner (0.2°)', 'binary_r')]:
        im = ax.imshow(img.T, cmap=cmap, origin='lower', aspect='auto')
        ax.set_title(ttl, fontsize=9)
        if cmap in ('viridis',): plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    p = os.path.join(save_dir, 'panel_segmentation.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close(); print(f"  Saved: {p}")

    # Porositas per-slice
    sl = np.arange(n_test)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(por_gt['porosity_per_slice']*100, sl, color='black', lw=1.5, label='GT (0.2°)')
    ax.plot(por_05['porosity_per_slice']*100, sl, color='royalblue', lw=1, label='U-Net thr=0.5')
    ax.plot(por_otsu['porosity_per_slice']*100, sl, color='tomato', lw=1, label='U-Net Otsu')
    ax.set_xlabel('Porositas (%)'); ax.set_ylabel('Slice (Z)')
    ax.set_title('Porositas per-slice'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); p = os.path.join(save_dir, 'panel_porosity.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close(); print(f"  Saved: {p}")

## 12. Main Pipeline

In [ ]:
# ============================================================================
# MAIN
# ============================================================================
def main():
    os.makedirs(cfg.SAVE_DIR, exist_ok=True)
    print("\n" + "="*65); print("  RESIDUAL U-NET SEGMENTATION — CONFIG"); print("="*65)
    print(f"  Training pair : {cfg.TRAINING_PAIR}")
    print(f"  Patch         : {cfg.PATCH_SIZE}²  ×{cfg.PATCHES_PER_SLICE}/slice")
    print(f"  Input 2.5D    : {cfg.UNET_IN_CH}-ch  ->  {cfg.N_CLASSES} kelas (softmax, class0=pori)")
    print(f"  Filters       : [32,64,128,256,512] (tetap, identik Blok B)")
    print(f"  Batch/Epochs  : {cfg.BATCH_SIZE}/{cfg.EPOCHS}  LR={cfg.LR}")
    print(f"  Loss          : Dice+CE (ce_w={cfg.CE_WEIGHT})")
    print(f"  AMP           : {cfg.USE_AMP}  Workers={cfg.NUM_WORKERS} (CPU={CPU_COUNT})")
    print(f"  Save dir      : {cfg.SAVE_DIR}")

    # sanity arsitektur 2.5D: input (B,5,H,W) -> output (B,2,H,W)
    _t = torch.randn(2, cfg.UNET_IN_CH, 256, 256).to(device)
    _m = ResidualUNet(in_ch=cfg.UNET_IN_CH, n_classes=cfg.N_CLASSES).to(device)
    _o = _m(_t)
    assert _o.shape == (2, cfg.N_CLASSES, 256, 256), f"Shape mismatch {_o.shape}"
    print(f"\n  Res-UNet 2.5D OK -- params: {count_parameters(_m):,}  out={tuple(_o.shape)}")
    del _t, _o, _m; torch.cuda.empty_cache()

    # ── LOAD ──────────────────────────────────────────────────────────────────
    print("\n" + "="*65); print("  LOADING DATA"); print("="*65)

    # Target biner (pori=1)
    if cfg.PATH_SEG is not None:
        seg_raw = load_and_crop_volume(cfg.PATH_SEG, cfg.CROP_SIZE, cfg.CROP_ORIGIN)
        seg_vol = (seg_raw > 0.5).astype(np.float32)
        seg_thresh = None
        print(f"  Target biner dari file: {os.path.basename(cfg.PATH_SEG)}")
    else:
        vol_02 = load_and_crop_volume(cfg.PATH_02, cfg.CROP_SIZE, cfg.CROP_ORIGIN)
        seg_vol, seg_thresh = binarize_grayscale(vol_02, cfg.PORE_IS_LOW, 'otsu')
        print(f"  Target biner via Otsu(0.2°): thresh={seg_thresh:.4f}, porositas GT={seg_vol.mean()*100:.2f}%")

    # Input noisy ternormalisasi
    inputs = []
    if cfg.TRAINING_PAIR in ('0.8', 'both'):
        v08 = load_and_crop_volume(cfg.PATH_08, cfg.CROP_SIZE, cfg.CROP_ORIGIN)
        v08_n, p1_08, p99_08 = normalize_volume(v08); inputs.append(('0.8', v08_n, p1_08, p99_08))
    if cfg.TRAINING_PAIR in ('0.4', 'both'):
        v04 = load_and_crop_volume(cfg.PATH_04, cfg.CROP_SIZE, cfg.CROP_ORIGIN)
        v04_n, p1_04, p99_04 = normalize_volume(v04); inputs.append(('0.4', v04_n, p1_04, p99_04))

    # Gabung: bila 'both', tumpuk sepanjang Z; target digandakan sejajar
    noisy_vol = np.concatenate([x[1] for x in inputs], axis=0)
    seg_full  = np.concatenate([seg_vol for _ in inputs], axis=0)
    norm_stats = {f'p1_{k}': float(p1) for k, _, p1, _ in inputs}
    norm_stats.update({f'p99_{k}': float(p99) for k, _, _, p99 in inputs})
    norm_stats['seg_thresh'] = seg_thresh
    np.save(os.path.join(cfg.SAVE_DIR, 'norm_stats.npy'), norm_stats)
    print(f"\n  Input gabungan: {noisy_vol.shape}  (sumber: {[k for k,_,_,_ in inputs]})")

    # ── SPLIT ──────────────────────────────────────────────────────────────────
    # Split berbasis volume tunggal asli supaya test set = irisan Z yang sama untuk
    # tiap sumber. Kita split pada panjang SATU volume lalu offset utk tiap blok.
    n_single = seg_vol.shape[0]
    tr1, va1, te1 = split_slices(n_single, cfg.TRAIN_FRAC, cfg.VAL_FRAC)
    train_idx, val_idx, test_idx = [], [], []
    for b in range(len(inputs)):
        off = b * n_single
        train_idx += [i+off for i in tr1]
        val_idx   += [i+off for i in va1]
        test_idx  += [i+off for i in te1]
    print(f"  Train/Val/Test slices : {len(train_idx)}/{len(val_idx)}/{len(test_idx)}")

    block_len = n_single   # panjang 1 volume sumber; jendela 2.5D tak menembus blok

    train_ds = SegPatchDataset(noisy_vol, seg_full, train_idx, cfg.PATCH_SIZE,
                               cfg.PATCHES_PER_SLICE, cfg.AUGMENT, 'train', block_len=block_len)
    val_ds   = SegPatchDataset(noisy_vol, seg_full, val_idx, cfg.PATCH_SIZE,
                               cfg.PATCHES_PER_SLICE, False, 'val', block_len=block_len)
    lk = dict(batch_size=cfg.BATCH_SIZE, num_workers=cfg.NUM_WORKERS,
              pin_memory=True, persistent_workers=(cfg.NUM_WORKERS > 0), prefetch_factor=2)
    train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **lk)
    val_loader   = DataLoader(val_ds, shuffle=False, **lk)
    print(f"  Train patches : {len(train_ds):,} ({len(train_loader):,} batches)")
    print(f"  Val patches   : {len(val_ds):,} ({len(val_loader):,} batches)")

    # ── TRACKER + TRAIN ────────────────────────────────────────────────────────
    tracker = ResourceTracker(cfg.TRACKER_INTERVAL, os.path.join(cfg.SAVE_DIR, 'resource_log.csv'))
    tracker.start()
    print("\n" + "="*65); print("  TRAINING RESIDUAL U-NET"); print("="*65)
    model, history, model_path = train_model(cfg, tracker, train_loader, val_loader, norm_stats)
    plot_training_history(history, cfg.SAVE_DIR)

    # ── INFERENCE ──────────────────────────────────────────────────────────────
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    print(f"\n  Loaded best: epoch={ckpt['epoch']} val_loss={ckpt['val_loss']:.6f} val_dice={ckpt.get('val_dice',float('nan')):.4f}")

    prob_vol, reg_metrics, psnr_list, ssim_list, rmse_list = \
        run_inference_testset(model, noisy_vol, seg_full, test_idx, cfg, device, tracker,
                              block_len=block_len)

    # ── DUA JALUR BINER ────────────────────────────────────────────────────────
    seg_test = seg_full[test_idx]
    bin_05   = (prob_vol >= 0.5).astype(np.uint8)                     # (1) segmentasi langsung
    otsu_t   = threshold_otsu(prob_vol.flatten())
    bin_otsu = (prob_vol >= otsu_t).astype(np.uint8)                  # (2) Otsu(prob) apple-to-apple
    print(f"\n  Otsu threshold pada peta probabilitas: {otsu_t:.4f}")

    seg_m_05   = seg_metrics_from_prob(prob_vol, seg_test, 0.5)
    seg_m_otsu = seg_metrics_from_prob(prob_vol, seg_test, otsu_t)
    print(f"  [thr=0.5 ] Dice={seg_m_05['Dice']:.4f}  IoU={seg_m_05['IoU']:.4f}  Acc={seg_m_05['Acc']:.4f}")
    print(f"  [Otsu    ] Dice={seg_m_otsu['Dice']:.4f}  IoU={seg_m_otsu['IoU']:.4f}  Acc={seg_m_otsu['Acc']:.4f}")

    # ── PETROPHYSICS (GT, thr0.5, Otsu) ────────────────────────────────────────
    print("\n" + "="*65); print("  PETROPHYSICAL ANALYSIS"); print("="*65)
    bin_gt = seg_test.astype(np.uint8)
    por_gt   = calculate_porosity(bin_gt)
    por_05   = calculate_porosity(bin_05)
    por_otsu = calculate_porosity(bin_otsu)
    ssa_gt   = calculate_ssa_volume(bin_gt, cfg.VOXEL_SIZE_UM)
    ssa_05   = calculate_ssa_volume(bin_05, cfg.VOXEL_SIZE_UM)
    ssa_otsu = calculate_ssa_volume(bin_otsu, cfg.VOXEL_SIZE_UM)

    vox_m = cfg.VOXEL_SIZE_UM*1e-6; D,H,W = bin_gt.shape; vol_m3 = D*H*W*vox_m**3
    def _Sv(b): return (b.astype(int)-binary_erosion(b).astype(int)).sum()*vox_m**2/vol_m3
    kc_gt   = kozeny_carman_volume(por_gt['porosity_total'],   _Sv(bin_gt),   cfg.KC_TORTUOSITY, cfg.KC_SHAPE_FACTOR)
    kc_05   = kozeny_carman_volume(por_05['porosity_total'],   _Sv(bin_05),   cfg.KC_TORTUOSITY, cfg.KC_SHAPE_FACTOR)
    kc_otsu = kozeny_carman_volume(por_otsu['porosity_total'], _Sv(bin_otsu), cfg.KC_TORTUOSITY, cfg.KC_SHAPE_FACTOR)

    with tracker.phase("petrophysics"):
        _, perm_gt   = run_petrophysics_parallel(bin_gt,   por_gt,   cfg.VOXEL_SIZE_UM, cfg.KC_TORTUOSITY, cfg.KC_SHAPE_FACTOR, "GT", tracker)
        _, perm_05   = run_petrophysics_parallel(bin_05,   por_05,   cfg.VOXEL_SIZE_UM, cfg.KC_TORTUOSITY, cfg.KC_SHAPE_FACTOR, "thr0.5", tracker)
        _, perm_otsu = run_petrophysics_parallel(bin_otsu, por_otsu, cfg.VOXEL_SIZE_UM, cfg.KC_TORTUOSITY, cfg.KC_SHAPE_FACTOR, "Otsu", tracker)

    # ── VISUAL ─────────────────────────────────────────────────────────────────
    visualize_results(noisy_vol[test_idx], prob_vol, seg_test,
                      bin_05, bin_otsu, bin_gt, seg_m_05, seg_m_otsu,
                      por_gt, por_05, por_otsu, cfg.SAVE_DIR)

    # ── SIMPAN ─────────────────────────────────────────────────────────────────
    print("\n" + "="*65); print("  MENYIMPAN OUTPUT"); print("="*65)
    imwrite(os.path.join(cfg.SAVE_DIR, 'prob_test_slices.tif'), prob_vol)
    imwrite(os.path.join(cfg.SAVE_DIR, 'seg_thr05_test.tif'), (bin_05*255).astype(np.uint8))
    imwrite(os.path.join(cfg.SAVE_DIR, 'seg_otsu_test.tif'), (bin_otsu*255).astype(np.uint8))

    df = pd.DataFrame({
        'slice_z': test_idx, 'PSNR_dB': psnr_list, 'SSIM': ssim_list, 'RMSE': rmse_list,
        'Porosity_GT_%':   por_gt['porosity_per_slice']*100,
        'Porosity_thr05_%': por_05['porosity_per_slice']*100,
        'Porosity_otsu_%':  por_otsu['porosity_per_slice']*100,
        'Perm_GT_mD': perm_gt, 'Perm_thr05_mD': perm_05, 'Perm_otsu_mD': perm_otsu,
    })
    df.to_csv(os.path.join(cfg.SAVE_DIR, 'metrics_per_slice.csv'), index=False)

    summary = {
        'model': 'Residual U-Net (segmentation)',
        'training_pair': cfg.TRAINING_PAIR, 'timestamp': datetime.now().isoformat(),
        'params': int(count_parameters(model)),
        'regression_metrics_prob_vs_GT': reg_metrics,
        'segmentation_metrics': {'thr0.5': seg_m_05, 'otsu': seg_m_otsu, 'otsu_threshold': float(otsu_t)},
        'porosity_%': {'GT': por_gt['porosity_percent'], 'thr0.5': por_05['porosity_percent'], 'otsu': por_otsu['porosity_percent']},
        'SSA_m2_per_cm3': {'GT': ssa_gt['SSA_m2_per_cm3'], 'thr0.5': ssa_05['SSA_m2_per_cm3'], 'otsu': ssa_otsu['SSA_m2_per_cm3']},
        'Permeability_mD_KC': {'GT': kc_gt['K_mD'], 'thr0.5': kc_05['K_mD'], 'otsu': kc_otsu['K_mD']},
    }
    with open(os.path.join(cfg.SAVE_DIR, 'results_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"  ✓ Semua output → {cfg.SAVE_DIR}")

    tracker.stop(); tracker.summary()
    tracker.plot(os.path.join(cfg.SAVE_DIR, 'resource_usage.png'))
    print("\n" + "="*65); print(f"  SELESAI — {cfg.SAVE_DIR}"); print("="*65)


## 13. Jalankan Pipeline
Eksekusi sel di bawah untuk memulai training → inference → petrophysics.

In [ ]:
import multiprocessing
if __name__ == '__main__':
    multiprocessing.freeze_support()
    main()